# EGCO461 Project: Thai Sentiment Classification (Baseline)

Notebook นี้เป็นจุดเริ่มต้น: โหลดข้อมูล ตัดคำภาษาไทย สร้าง Naïve Bayes baseline วัดผลด้วย **macro-F1** และสร้างไฟล์ส่งผล

หน้าที่ของกลุ่มคือทำให้ดีกว่า baseline นี้ และอธิบายได้ว่า **อะไรช่วยได้จริง เพราะอะไร**

In [42]:
# !pip install pythainlp scikit-learn pandas  # run once if needed
import pandas as pd
from pythainlp.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score, classification_report, confusion_matrix

## 1. Load data

In [43]:
train = pd.read_csv('data/train.csv')
valid = pd.read_csv('data/valid.csv')
sample = pd.read_csv('data/sample_labeled.csv')   # 100 labeled examples in the same style as the test set
test = pd.read_csv('data/test.csv')               # no labels
print(train.shape, valid.shape, sample.shape, test.shape)
train['category'].value_counts()

(24020, 3) (2669, 3) (100, 3) (1577, 2)


category
neu    13098
neg     6138
pos     4286
q        498
Name: count, dtype: int64

สังเกตว่าคลาสไม่สมดุลมาก (q มีน้อย) จึงใช้ macro-F1 ซึ่งให้น้ำหนักทุกคลาสเท่ากัน

In [44]:
train.sample(10, random_state=0)

,id,text,category
2128,tr02128,บ้านเราไม่ผ่านเพราะไรป๋า,q
12423,tr12423,กานิเย่เอจ ซุปเปอร์เอสเซ้น,neu
888,tr00888,เทคนิคการป้องกันผิวรับซัมเมอร์ คือ ทาครีมกันแด...,neu
10450,tr10450,โรงพยาบาลสมิติเวช ร่วมกับ สโมสรบุรีรัมย์ ยูไนเ...,neu
18410,tr18410,แก้ปัญหาที่ถูกต้องคือไม่สร้างทางเลือกให้อีกจะด...,neu
9382,tr09382,ยังงัยก็รัก CRV เหมือนเดิม แค่ให้ใหญ่ขึ้นรองรั...,neu
7551,tr07551,เช่นกันค่ะ ยังไม่ได้ของเหมือนกัน เสียค่าส่งอีก...,neg
8305,tr08305,สรุป อีนาวาร่าตัวซวย ทำให้คนทั้งตลาดตกงาน,neu
7346,tr07346,จนป่านนี้ผมยังรักษาไม่จบเลย ผมโดนหมากัด ฉีดวัค...,neu
15345,tr15345,นั่นสิ รออยู่เนี้ยะ,neu


## 2. Tokenize Thai text

ภาษาไทยไม่มีช่องว่างระหว่างคำ ต้องตัดคำก่อน (ใช้ `newmm` ของ PyThaiNLP)

In [45]:
def thai_tokenize(text):
    return word_tokenize(str(text), engine='newmm', keep_whitespace=False)

thai_tokenize('อร่อยมากกก ร้านนี้ต้องมาอีก 555')

['อร่อย', 'มาก', 'กก', 'ร้าน', 'นี้', 'ต้อง', 'มา', 'อีก', '555']

## 3. Baseline: bag of words + Multinomial Naïve Bayes

In [46]:
vectorizer = CountVectorizer(tokenizer=thai_tokenize, token_pattern=None)
X_train = vectorizer.fit_transform(train['text'])
X_valid = vectorizer.transform(valid['text'])
X_sample = vectorizer.transform(sample['text'])

model = MultinomialNB(alpha=1.0)   # Laplace smoothing, same idea as in the lecture
model.fit(X_train, train['category'])

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


## 4. Evaluate

In [47]:
from IPython.display import display

labels = ['pos', 'neu', 'neg', 'q']
pred_valid = model.predict(X_valid)
pred_sample = model.predict(X_sample)

print(f"valid: macro-F1 = {f1_score(valid['category'], pred_valid, labels=labels, average='macro'):.3f}")
print(f"sample: macro-F1 = {f1_score(sample['category'], pred_sample, labels=labels, average='macro'):.3f}")

print('\nValid classification report:')
print(classification_report(valid['category'], pred_valid, labels=labels, digits=3, zero_division=0))
print('Valid confusion matrix (Rows = True, Columns = Predicted):')
display(pd.DataFrame(
    confusion_matrix(valid['category'], pred_valid, labels=labels),
    index=[f'true_{l}' for l in labels],
    columns=[f'pred_{l}' for l in labels]
))

print('\nSample classification report:')
print(classification_report(sample['category'], pred_sample, labels=labels, digits=3, zero_division=0))
print('Sample confusion matrix (Rows = True, Columns = Predicted):')
display(pd.DataFrame(
    confusion_matrix(sample['category'], pred_sample, labels=labels),
    index=[f'true_{l}' for l in labels],
    columns=[f'pred_{l}' for l in labels]
))


valid: macro-F1 = 0.446
sample: macro-F1 = 0.483

Valid classification report:
              precision    recall  f1-score   support

         pos      0.500     0.393     0.440       476
         neu      0.784     0.606     0.684      1456
         neg      0.522     0.894     0.659       682
           q      0.000     0.000     0.000        55

    accuracy                          0.629      2669
   macro avg      0.452     0.473     0.446      2669
weighted avg      0.650     0.629     0.620      2669

Valid confusion matrix (Rows = True, Columns = Predicted):


,pred_pos,pred_neu,pred_neg,pred_q
true_pos,187,152,137,0
true_neu,169,883,404,0
true_neg,16,56,610,0
true_q,2,35,18,0



Sample classification report:
              precision    recall  f1-score   support

         pos      0.706     0.571     0.632        21
         neu      0.707     0.604     0.652        48
         neg      0.524     0.846     0.647        26
           q      0.000     0.000     0.000         5

    accuracy                          0.630       100
   macro avg      0.484     0.505     0.483       100
weighted avg      0.624     0.630     0.614       100

Sample confusion matrix (Rows = True, Columns = Predicted):


,pred_pos,pred_neu,pred_neg,pred_q
true_pos,12,6,3,0
true_neu,3,29,16,0
true_neg,2,2,22,0
true_q,0,4,1,0


**คำถามชวนคิด:** ทำไม recall ของ `pos` และ `q` ต่ำมาก? ลองดูตัวอย่างที่ทายผิดในเซลล์ถัดไป

In [48]:
pred = model.predict(X_valid)
errors = valid.assign(pred=pred)[lambda d: d['category'] != d['pred']]
errors.sample(15, random_state=1)[['text', 'category', 'pred']]

,text,category,pred
829,ภาพเล่าเรื่องทุกอย่าง ความจริงเป็นยังไงทุกคนรู...,neu,neg
1504,Nissan ไม่แพง,pos,neg
1803,เย็ดเข้ เป็ด+เอ็มเค อร่อยมาก ใครมาก็คงอยากกิน,pos,neg
1890,ออนิวดีแมคราคาเท่าไหร่คับ,neu,neg
2565,เบียร์ช้างขวดละ10 จ้า ขายถูกๆมีแต่ขวดด,neu,neg
880,คุณแม่,pos,neu
1259,จะกินฮอตพอท,neu,neg
2624,ว่างปะ,neu,pos
2420,เคยบอกกับตัวเองว่าสักวันต้องมีเบ้นขับเปนของตัว...,neu,neg
2422,ไม่ๆ พี่จ่ายเต็ม พวกเเกไปหารกัน 4 คน,pos,neg


## 5. Create a submission file

ไฟล์ต้องมีคอลัมน์ `id,category` ครบทุก id ใน `test.csv` ตั้งชื่อไฟล์ให้สื่อถึงโมเดล เช่น `G05_run1_nb.csv` (ส่งได้สูงสุด 5 ไฟล์)

In [49]:
submission = pd.DataFrame({'id': test['id'], 'category': model.predict(vectorizer.transform(test['text']))})
submission.to_csv('G00_run1_nb_baseline.csv', index=False)
submission['category'].value_counts()

category
neu    723
neg    635
pos    219
Name: count, dtype: int64

## Ideas to try (เลือกเองว่าจะลองอะไร แล้ววัดผลให้ชัด)

- TF-IDF, n-grams (`ngram_range=(1, 2)`), character n-grams
- Logistic Regression / Linear SVM, `class_weight='balanced'`
- จัดการข้อความ: ตัวอักษรซ้ำ (มากกกก), 555, อีโมจิ, ลิงก์
- ปรับ threshold หรือ resampling เพื่อช่วยคลาสเล็ก
- Pretrained Thai models เช่น WangchanBERTa (ต้องใช้ GPU เช่น Google Colab)
- **สำคัญ:** ใช้ `valid.csv` และ `sample_labeled.csv` เลือกโมเดล อย่าเดาจาก test